In [0]:

# =============================================================================
# NOTEBOOK: extract_team_names
# =============================================================================
#
# PURPOSE
# -------
# Discover football clubs in Bronze that are not yet represented in
# silver.team_mappings.
#
# This notebook is a PRE-SILVER data-quality gate.
#
# WHY BRONZE?
# -----------
# The existing Silver transformations deliberately fail when a source team
# does not have a canonical team_key.
#
# Therefore, we must discover new teams BEFORE running those transformations.
# Reading only Silver would miss teams whose records could not be processed.
#
# WHEN TO RUN
# -----------
# 1. Complete ingestion for football-data.org, FBref and football-data.co.uk.
# 2. Run this notebook.
# 3. Follow the instructions printed at the end.
#
# POSSIBLE OUTCOMES
# -----------------
# A. Zero unmapped teams:
#    No manual action is required. Proceed to the Silver notebooks.
#
# B. One or more unmapped teams:
#    Review the results and prepare approved mapping additions.
#    Do NOT run the dependent Silver notebooks until those are loaded.
#
# OUTPUT
# ------
# A diagnostic CSV directory:
#   .../team_standardization_review/unmapped_source_teams/
#
# The output is NOT automatically approved and must NOT be directly merged
# into silver.team_mappings.
#
# IMPORTANT
# ---------
# This notebook never creates canonical keys or guesses team identities.
# Existing mappings and canonical team keys remain unchanged.
# =============================================================================

from functools import reduce

from pyspark.sql import functions as F
from pyspark.sql.types import StructType


# =============================================================================
# 1. Configuration
# =============================================================================

CATALOG = "football_data_platform"

BRONZE_SCHEMA = f"{CATALOG}.bronze"
SILVER_SCHEMA = f"{CATALOG}.silver"

TEAM_MAPPINGS_TABLE = f"{SILVER_SCHEMA}.team_mappings"
CANONICAL_TEAMS_TABLE = f"{SILVER_SCHEMA}.canonical_teams"

REVIEW_PATH = (
    "/Volumes/football_data_platform/raw/source_files/"
    "team_standardization_review"
)

EXPORT_PATH = f"{REVIEW_PATH}/unmapped_source_teams"

API_SOURCE = "football-data.org"
FBREF_SOURCE = "fbref"
CSV_SOURCE = "football-data.co.uk"

# Matches the current five-league scope.
#
# FBref Bronze naming:
#   fbref_{dataset}_{league_code}
#
# football-data.co.uk Bronze naming:
#   football_data_{league_name}

FBREF_DATASETS = [
    "standard_stats",
    "shooting_stats",
    "goalkeeping_stats",
    "playing_time",
    "miscellaneous",
    "standings",
]

FBREF_LEAGUES = [
    "pl",
    "ll",
    "bl",
    "sa",
    "lo",
]

FOOTBALL_DATA_LEAGUES = [
    "premier_league",
    "bundesliga",
    "serie_a",
    "la_liga",
    "ligue_one",
]


# =============================================================================
# 2. Source configuration
# =============================================================================
#
# Each configuration identifies:
#   - Bronze table
#   - source system
#   - source name field
#   - source ID field, when available
#
# football-data.org exposes team IDs.
# FBref and football-data.co.uk are matched by source team names.
#
# The nested API fields match the current Bronze schemas used by
# load_api_team_to_silver, load_api_matches_to_silver and
# load_api_standings_to_silver.
# =============================================================================

SOURCE_TABLES = [
    (
        API_SOURCE,
        "football_data_org_teams",
        "name",
        "id",
    ),
    (
        API_SOURCE,
        "football_data_org_standings",
        "team.name",
        "team.id",
    ),
    (
        API_SOURCE,
        "football_data_org_matches",
        "homeTeam.name",
        "homeTeam.id",
    ),
    (
        API_SOURCE,
        "football_data_org_matches",
        "awayTeam.name",
        "awayTeam.id",
    ),
]

for dataset in FBREF_DATASETS:
    for league in FBREF_LEAGUES:
        SOURCE_TABLES.append(
            (
                FBREF_SOURCE,
                f"fbref_{dataset}_{league}",
                "Squad",
                None,
            )
        )

for league in FOOTBALL_DATA_LEAGUES:
    SOURCE_TABLES.extend(
        [
            (
                CSV_SOURCE,
                f"football_data_{league}",
                "HomeTeam",
                None,
            ),
            (
                CSV_SOURCE,
                f"football_data_{league}",
                "AwayTeam",
                None,
            ),
        ]
    )


# =============================================================================
# 3. Helper: verify nested fields
# =============================================================================

def bronze_field_exists(df, column_path):
    """
    Check whether a top-level or nested field exists.

    Examples:
        name
        team.id
        homeTeam.name
    """

    current_type = df.schema

    for field_name in column_path.split("."):

        if not isinstance(current_type, StructType):
            return False

        if field_name not in current_type.fieldNames():
            return False

        current_type = current_type[field_name].dataType

    return True


# =============================================================================
# 4. Helper: extract a source team observation
# =============================================================================

def extract_source_teams(
    source_system,
    table_name,
    name_column,
    id_column,
):
    """
    Extract source-specific team identities from a Bronze table.

    Keeps the provider's original identity:
      - source_system
      - source_team_id, when available
      - source_team_name

    Also preserves source-table metadata for investigation.
    """

    table = f"{BRONZE_SCHEMA}.{table_name}"

    if not spark.catalog.tableExists(table):
        raise ValueError(
            f"Required Bronze table does not exist: {table}"
        )

    bronze_df = spark.table(table)

    if not bronze_field_exists(bronze_df, name_column):
        raise ValueError(
            f"Missing team name field {name_column} in {table}"
        )

    if id_column and not bronze_field_exists(bronze_df, id_column):
        raise ValueError(
            f"Missing team ID field {id_column} in {table}"
        )

    source_team_id = (
        F.trim(F.col(id_column).cast("string"))
        if id_column
        else F.lit(None).cast("string")
    )

    return (
        bronze_df
        .select(
            F.lit(source_system).alias("source_system"),

            source_team_id.alias("source_team_id"),

            F.trim(
                F.col(name_column).cast("string")
            ).alias("source_team_name"),

            F.lit(table_name).alias("source_table"),
        )
    )


# =============================================================================
# 5. Validate reference tables
# =============================================================================

print("=" * 70)
print("TEAM STANDARDIZATION — SOURCE IDENTITY DISCOVERY")
print("=" * 70)

print("\nSTEP 1: Checking reference tables.")

for table in [
    TEAM_MAPPINGS_TABLE,
    CANONICAL_TEAMS_TABLE,
]:
    if not spark.catalog.tableExists(table):
        raise ValueError(
            f"Missing reference table: {table}. "
            "Complete the initial reference-data setup first."
        )

print("Reference tables exist.")


# =============================================================================
# 6. Read team identities from Bronze
# =============================================================================

print("\nSTEP 2: Extracting team identities from Bronze.")

team_dataframes = []

for (
    source_system,
    table_name,
    name_column,
    id_column,
) in SOURCE_TABLES:

    team_dataframes.append(
        extract_source_teams(
            source_system,
            table_name,
            name_column,
            id_column,
        )
    )

all_teams_df = reduce(
    lambda left, right: left.unionByName(right),
    team_dataframes,
)

# Do not silently ignore malformed names.
invalid_names_df = all_teams_df.filter(
    F.col("source_team_name").isNull()
    | (F.col("source_team_name") == "")
)

if invalid_names_df.limit(1).count() > 0:

    display(
        invalid_names_df
        .select("source_system", "source_table")
        .distinct()
    )

    raise ValueError(
        "Bronze contains missing or blank team names. "
        "Investigate the source data before continuing."
    )

# football-data.org must always have a source team ID.
invalid_api_ids_df = all_teams_df.filter(
    (F.col("source_system") == API_SOURCE)
    & (
        F.col("source_team_id").isNull()
        | (F.col("source_team_id") == "")
    )
)

if invalid_api_ids_df.limit(1).count() > 0:

    display(invalid_api_ids_df.limit(20))

    raise ValueError(
        "football-data.org Bronze contains missing team IDs."
    )


# =============================================================================
# 7. Create source-specific identity keys
# =============================================================================
#
# API identity:
#   source_system + source_team_id
#
# FBref / football-data.co.uk identity:
#   source_system + source_team_name
#
# These rules must remain consistent with the existing Silver resolvers.
# =============================================================================

observed_df = all_teams_df.withColumn(
    "identity_key",
    F.when(
        F.col("source_system") == API_SOURCE,
        F.col("source_team_id"),
    ).otherwise(
        F.col("source_team_name")
    ),
)

# Aggregate observations so the same source team is reviewed only once.
#
# An API ID can have more than one observed display name. That should
# not create multiple identities because its provider ID is authoritative.

distinct_teams_df = (
    observed_df
    .groupBy(
        "source_system",
        "identity_key",
    )
    .agg(
        F.min("source_team_id").alias("source_team_id"),

        F.min("source_team_name").alias("source_team_name"),

        F.sort_array(
            F.collect_set("source_team_name")
        ).alias("observed_names"),

        F.sort_array(
            F.collect_set("source_table")
        ).alias("source_tables"),
    )
)


# =============================================================================
# 8. Read existing mappings
# =============================================================================

print("\nSTEP 3: Comparing observations with existing mappings.")

existing_mappings_df = (
    spark.table(TEAM_MAPPINGS_TABLE)
    .select(
        "source_system",
        F.trim(
            F.col("source_team_id").cast("string")
        ).alias("source_team_id"),
        F.trim(
            F.col("source_team_name")
        ).alias("source_team_name"),
        "team_key",
    )
    .withColumn(
        "identity_key",
        F.when(
            F.col("source_system") == API_SOURCE,
            F.col("source_team_id"),
        ).otherwise(
            F.col("source_team_name")
        ),
    )
)

invalid_existing_df = existing_mappings_df.filter(
    F.col("identity_key").isNull()
    | (F.col("identity_key") == "")
    | F.col("team_key").isNull()
)

if invalid_existing_df.limit(1).count():
    raise ValueError(
        "Existing team mappings contain missing identities or keys."
    )

duplicate_existing_df = (
    existing_mappings_df
    .groupBy("source_system", "identity_key")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_existing_df.limit(1).count():
    raise ValueError(
        "Existing mappings contain duplicate source identities."
    )

existing_identities_df = (
    existing_mappings_df
    .select(
        "source_system",
        "identity_key",
    )
    .distinct()
)


# =============================================================================
# 9. Identify new, unmapped teams
# =============================================================================

unmapped_teams_df = (
    distinct_teams_df
    .join(
        existing_identities_df,
        on=["source_system", "identity_key"],
        how="left_anti",
    )
    .select(
        "source_system",
        "source_team_id",
        "source_team_name",
        "observed_names",
        "source_tables",
    )
    .orderBy(
        "source_system",
        "source_team_name",
    )
)

total_observed = distinct_teams_df.count()
unmapped_count = unmapped_teams_df.count()

print(f"Distinct observed source identities: {total_observed}")
print(f"Unmapped source identities: {unmapped_count}")

print("\nSource breakdown:")

display(
    unmapped_teams_df
    .groupBy("source_system")
    .count()
    .orderBy("source_system")
)


# =============================================================================
# 10. Export unmapped identities for human review
# =============================================================================
#
# This is an automated diagnostic export, NOT an approved mapping file.
#
# The result can be downloaded or inspected directly in Unity Catalog.
# It is overwritten on each discovery run and should not be manually edited.
# =============================================================================

export_df = (
    unmapped_teams_df
    .withColumn(
        "observed_names",
        F.concat_ws("|", F.col("observed_names")),
    )
    .withColumn(
        "source_tables",
        F.concat_ws("|", F.col("source_tables")),
    )
)

(
    export_df
    .coalesce(1)
    .write
    .format("csv")
    .mode("overwrite")
    .option("header", True)
    .save(EXPORT_PATH)
)

print(f"\nDiscovery export: {EXPORT_PATH}")
print("The CSV is stored as a part-*.csv file in that directory.")


# =============================================================================
# 11. Final instructions for the user
# =============================================================================

print("\n" + "=" * 70)
print("TEAM DISCOVERY RESULT")
print("=" * 70)

if unmapped_count == 0:

    print("\nSUCCESS: All observed source teams have mappings.")

    print("\nNEXT STEPS:")
    print("1. No new canonical teams or mappings are required.")
    print("2. Do NOT run the mapping loader unnecessarily.")
    print("3. Proceed with the existing Silver transformation notebooks.")
    print("4. No changes to the Silver code are required.")

else:

    print(
        f"\nACTION REQUIRED: {unmapped_count} unmapped "
        "source identities detected."
    )

    display(unmapped_teams_df)

    print("\nNEXT STEPS:")
    print("1. Review each unmapped identity above.")
    print("2. Check silver.canonical_teams for an existing club.")
    print("3. If it exists, reuse its exact canonical team_name.")
    print("4. If it is genuinely new, approve a new canonical club.")
    print("5. Prepare these two UTF-8 CSV files:")
    print("   - canonical_teams_additions.csv")
    print("   - team_mappings_additions.csv")
    print("6. Upload both files to:")
    print(f"   {REVIEW_PATH}")
    print("7. Run load_team_mappings_to_silver.")
    print("8. Rerun this discovery notebook.")
    print("9. Once zero unmapped identities remain, run Silver.")

    print("\nREQUIRED FILE FORMAT:")

    print("\ncanonical_teams_additions.csv")
    print("team_name")
    print("Example New Club FC")

    print("\nteam_mappings_additions.csv")
    print(
        "source_system,source_team_id,"
        "source_team_name,canonical_team_name"
    )

    print("\nIMPORTANT RULES:")
    print("- Never invent or reuse a canonical team_key.")
    print("- The loader will allocate keys for approved new clubs.")
    print("- Reuse existing canonical names for new aliases.")
    print("- football-data.org mappings require source_team_id.")
    print("- FBref and football-data.co.uk source IDs remain blank.")
    print("- Do not run dependent Silver notebooks until resolved.")

print("\nDiscovery completed.")
